# OV cell fractions by expression subtype

Load plotting settings, subtype order, colors, and the fixed server output path.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from scipy.stats import mannwhitneyu
import matplotlib as mpl
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")
mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans"],
        "font.size": 8,
        "axes.titlesize": 9,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "axes.linewidth": 0.7,
        "xtick.major.width": 0.7,
        "ytick.major.width": 0.7,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "savefig.edgecolor": "white",
        "savefig.dpi": 300,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)
plots_dir = "/Path/to/output/cancer/OV/plots_gene_expression_subtype"
methods = ["SABER", "DWLS", "BayesPrism"]
npg_colors = ["#3C5488", "#E64B35", "#00A087", "#4DBBD5", "#F39B7F"]
subtype_col = "gene_expression_subtype"
subtype_order = ["differentiated", "immunoreactive", "mesenchymal", "proliferative"]
subtype_palette = dict(zip(subtype_order, npg_colors))


## Test predicted fractions across OV expression subtypes

Match TCGA patients to SABER, DWLS, and BayesPrism predictions, compare each cell type’s highest-mean subtype with the other groups using patients as statistical units, and save per-method PDFs and statistics CSV files.


In [2]:
prediction_paths = {method: os.path.join("/Path/to/cancer/results/OV1", f"{method}.csv") for method in methods}
missing_prediction_files = [path for path in prediction_paths.values() if not os.path.exists(path)]
if missing_prediction_files:
    raise FileNotFoundError(f"Missing required prediction files: {missing_prediction_files}")

os.makedirs(plots_dir, exist_ok=True)
subtype_df = pd.read_csv("/Path/to/cancer/data/OV1/TCGA_OV_gene_expression_subtype", sep="\t")
subtype_df = subtype_df[["sample", subtype_col]].dropna()
subtype_df["sample"] = subtype_df["sample"].astype(str).str.replace(".", "-", regex=False)
subtype_df = subtype_df[subtype_df["sample"].str[13:15].eq("01")].copy()
subtype_df["patient"] = subtype_df["sample"].str[:12]
assert not subtype_df["patient"].duplicated().any()
subtype_df = subtype_df.set_index("patient")
subtype_df = subtype_df[subtype_df[subtype_col].isin(subtype_order)]
all_methods_stats = []
for method in methods:
    pred_df = pd.read_csv(prediction_paths[method], index_col=0)
    pred_df.index = pred_df.index.astype(str).str.replace(".", "-", regex=False).str[:12]
    common_samples = pred_df.index.intersection(subtype_df.index).drop_duplicates()
    base_plot_data = pd.DataFrame(
        {"Sample": common_samples, "Subtype": subtype_df.loc[common_samples, subtype_col].values}
    ).dropna()
    base_plot_data["Subtype"] = pd.Categorical(
        base_plot_data["Subtype"], categories=subtype_order, ordered=True
    )
    base_plot_data = base_plot_data.sort_values("Subtype")
    subtype_counts = base_plot_data["Subtype"].value_counts(sort=False).reindex(subtype_order, fill_value=0)
    fig, axes = plt.subplots(1, 3, figsize=(4.5, 3.4), sharey=False)
    axes = axes.flatten()
    method_stats_rows = []
    for ax, (cell_type, feature_cols) in zip(
        axes,
        {
            "Fibroblasts": "Fibroblasts",
            "Epithelial cells": "Epithelial_cells",
            "Immune": ["T_NK_cells", "B_cells", "Plasma_cells", "Myeloid_cells"],
        }.items(),
    ):
        plot_data = base_plot_data.copy()
        if isinstance(feature_cols, list):
            plot_data["Proportion"] = pred_df.loc[plot_data["Sample"], feature_cols].sum(axis=1).values
        else:
            plot_data["Proportion"] = pred_df.loc[plot_data["Sample"], feature_cols].values
        plot_data = plot_data.dropna(subset=["Subtype", "Proportion"])
        subtype_means = plot_data.groupby("Subtype", observed=False)["Proportion"].mean()
        reference_subtype = subtype_means.idxmax()
        reference_values = plot_data.loc[plot_data["Subtype"] == reference_subtype, "Proportion"]
        stats_rows = []
        for comparison_subtype in subtype_order:
            if comparison_subtype == reference_subtype:
                continue
            comparison_values = plot_data.loc[plot_data["Subtype"] == comparison_subtype, "Proportion"]
            assert len(reference_values) > 2 and len(comparison_values) > 2
            u_stat, p_value = mannwhitneyu(reference_values, comparison_values, alternative="two-sided")
            row = {
                "method": method,
                "cell_type": cell_type,
                "reference_subtype": reference_subtype,
                "comparison_subtype": comparison_subtype,
                "n_reference": len(reference_values),
                "n_comparison": len(comparison_values),
                "mean_reference": reference_values.mean(),
                "mean_comparison": comparison_values.mean(),
                "u_statistic": u_stat,
                "p_value": p_value,
                "significance": "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns",
            }
            stats_rows.append(row)
            method_stats_rows.append(row)
            all_methods_stats.append(row)
        sns.boxplot(
            x="Subtype",
            y="Proportion",
            data=plot_data,
            order=subtype_order,
            palette=subtype_palette,
            ax=ax,
            width=0.58,
            linewidth=0.8,
            fliersize=0,
            saturation=0.95,
            boxprops=dict(edgecolor="black", linewidth=0.8),
            whiskerprops=dict(color="black", linewidth=0.8),
            capprops=dict(color="black", linewidth=0.8),
            medianprops=dict(color="black", linewidth=1.0),
        )
        sns.stripplot(
            x="Subtype",
            y="Proportion",
            data=plot_data,
            order=subtype_order,
            ax=ax,
            color="black",
            size=1.8,
            alpha=0.35,
            jitter=0.22,
            linewidth=0,
        )
        ax.set_title(cell_type, fontsize=9, fontweight="normal", pad=5)
        ax.set_xlabel("")
        ax.set_ylabel(f"{method}-predicted proportion")
        data_min = plot_data["Proportion"].min()
        data_max = plot_data["Proportion"].max()
        y_range = data_max - data_min
        assert y_range != 0
        h = y_range * 0.05
        line_y = data_max + h
        final_y = line_y
        x_reference = subtype_order.index(reference_subtype)
        for row in stats_rows:
            sig = row["significance"]
            if sig == "ns":
                continue
            x_comparison = subtype_order.index(row["comparison_subtype"])
            ax.plot(
                [x_reference, x_reference, x_comparison, x_comparison],
                [line_y, line_y + h / 2, line_y + h / 2, line_y],
                lw=0.8,
                c="black",
            )
            ax.text(
                (x_reference + x_comparison) / 2,
                line_y + h / 2,
                sig,
                ha="center",
                va="bottom",
                fontsize=8,
                color="black",
            )
            final_y = line_y + h / 2
            line_y += h * 1.6
        ax.set_ylim(data_min - y_range * 0.08, final_y + y_range * 0.2)
        ax.tick_params(axis="x", which="both", labelbottom=False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(0.7)
        ax.spines["bottom"].set_linewidth(0.7)
        ax.tick_params(axis="both", which="major", width=0.7, length=3, pad=2)
        ax.grid(False)
    pd.DataFrame(method_stats_rows).to_csv(
        os.path.join(plots_dir, f"{method}_OV1_fibroblast_epi_immune_by_gene_expression_subtype_stats.csv"),
        index=False,
    )
    fig.suptitle(
        f"{method} fibroblast, epithelial, and immune fractions by OV gene expression subtype",
        fontsize=11,
        fontweight="normal",
        y=1.03,
    )
    fig.legend(
        handles=[
            Patch(
                facecolor=subtype_palette[subtype],
                edgecolor="black",
                linewidth=0.5,
                label=f"{subtype} (n={int(subtype_counts[subtype])})",
            )
            for subtype in subtype_order
        ],
        loc="lower center",
        ncol=len(subtype_order),
        frameon=False,
        bbox_to_anchor=(0.5, 0.01),
        handlelength=1.1,
        handleheight=0.8,
        handletextpad=0.5,
        columnspacing=1.2,
        fontsize=7,
    )
    plt.tight_layout(rect=[0, 0.1, 1, 1])
    plt.savefig(
        os.path.join(plots_dir, f"{method}_OV1_fibroblast_epi_immune_by_gene_expression_subtype_boxplot.pdf"),
        bbox_inches="tight",
    )
    plt.close()
pd.DataFrame(all_methods_stats).to_csv(
    os.path.join(plots_dir, "all_methods_OV1_fibroblast_epi_immune_by_gene_expression_subtype_stats.csv"),
    index=False,
)
